
  ### Objective
  This Project is to develop an end to end ML regression system that predict the price of a residential properties.
  

In [ ]:
# # install Libraries

In [ ]:
# Import all imp libraries
import os
import glob
import joblib
import gradio as gr
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
###################################################
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import(
    StandardScaler,
    OneHotEncoder,
    PolynomialFeatures
)
from sklearn.feature_selection import SelectKBest , f_regression
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.metrics import(
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [ ]:
!pip install -q kaggle

In [ ]:
!pip install --upgrade kaggle

In [ ]:
!mkdir -p ~/.kaggle
!echo '{"username":"chetanpadwal","key":"KGAT_7aed99165cf14cd1032f22b338bb1a8f"}'>~/.kaggle/kaggle.json

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)

!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json


In [ ]:
# test kaggle API
!kaggle datasets list -s "house price prediction"


In [ ]:
!kaggle competitions list -s "indian urban house price"

In [ ]:
import zipfile
import glob
import os

# Look for zip files specifically in the 'house_price_data' directory
zip_files = glob.glob("house_price_data/*.zip")

if not zip_files:
    print("No zip files found in 'house_price_data'. Please ensure the dataset was downloaded correctly.")
else:
    for zip_file in zip_files:
        print("Extracting:", zip_file)

        with zipfile.ZipFile(zip_file, "r") as zip_ref:
            # Extract to the parent directory to make CSVs accessible at house_price_data/*
            zip_ref.extractall(os.path.dirname(zip_file))

    print("Extraction completed!")

In [ ]:
import glob

csv_files = glob.glob(
    "house_price_data/**/*.csv",
    recursive=True
)

print("CSV files found:")

for file in csv_files:
    print(file)

In [ ]:
!kaggle competitions download -c indian-urban-house-price-prediction -p house_price_data

In [ ]:
import pandas as pd

train_files = [
    file for file in csv_files
    if "train" in file.lower()
]

if not train_files:
    print("Error: No training CSV file found. This likely means the dataset was not downloaded or extracted correctly.")
    print("Please ensure you have downloaded the dataset (e.g., using `!kaggle competitions download -c indian-urban-house-price-prediction -p house_price_data`) and extracted it before running this cell again.")
    df = None # Set df to None to avoid further NameError if display(df.head()) is called
else:
    train_file = train_files[0]
    df = pd.read_csv(train_file)

    print("Dataset loaded successfully!")
    print("Shape:", df.shape)

    if df is not None:
        display(df.head())

In [ ]:
df=pd.read_csv('/content/indian_house_prices_dataset.csv')
df

In [ ]:
# Dataset Information
print('Dataset Shape :')
print(df.shape)
print('\nDataset columns: =')
print(df.columns.tolist())
print('\n Data Types ')
display(df.dtypes)


In [ ]:
print('\n Missing Values : ')
display(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
# stat summary :
display(df.describe())

In [ ]:
# duplicate records
print('Number of duplicate Rows :',df.duplicated().sum())


In [ ]:
# distribution
plt.figure(figsize=(10,5))
sns.histplot(
    df['Price_INR_Lakhs'],
    kde=True

)

In [ ]:
# Bosplot of Target
plt.figure(figsize=(10,5))
sns.boxplot(
    df['Price_INR_Lakhs']
)

In [ ]:
x_temp = df.drop(columns=['Price_INR_Lakhs'])
y = df['Price_INR_Lakhs']

numeric_features = x_temp.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = x_temp.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)


In [ ]:
# Univariate Numerical Analysis
for col in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(
        df[col],
        kde=True
    )
    plt.title(f"Distribution of {col}")
    plt.show()


In [ ]:
for col in categorical_features:
    if df[col].nunique() <= 20:
        plt.figure(figsize=(10, 5))
        df[col].value_counts().plot(
            kind="bar"
        )
        plt.title(f"Distribution of {col}")
        plt.xticks(rotation=45)
        plt.show()

In [ ]:
# correlation analysis
plt.figure(figsize=(14, 10))
corr = df[numeric_features + ['Price_INR_Lakhs']].corr()
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)
plt.title("Correlation Heatmap")

plt.show()

In [ ]:
df_fe = df.copy()

# Detect possible area column
area_cols = [
    col for col in df_fe.columns
    if any(word in col.lower()
           for word in ["area", "sqft", "square"])
]
# Detect bedroom column
bedroom_cols = [
    col for col in df_fe.columns
    if any(word in col.lower()
           for word in ["bedroom", "bhk"])
]
# Detect bathroom column
bathroom_cols = [
    col for col in df_fe.columns
    if any(word in col.lower()
           for word in ["bathroom", "bath"])
]
print("Area columns:", area_cols)
print("Bedroom columns:", bedroom_cols)
print("Bathroom columns:", bathroom_cols)

In [ ]:
df_fe = df.copy()

# Detect possible area column
area_cols = [
    col for col in df_fe.columns
    if any(word in col.lower()
           for word in ["area", "sqft", "square"])
]

# Detect bedroom column
bedroom_cols = [
    col for col in df_fe.columns
    if any(word in col.lower()
           for word in ["bedroom", "bhk"])
]

# Detect bathroom column
bathroom_cols = [
    col for col in df_fe.columns
    if any(word in col.lower()
           for word in ["bathroom", "bath"])
]

print("Area columns:", area_cols)
print("Bedroom columns:", bedroom_cols)
print("Bathroom columns:", bathroom_cols)

In [ ]:
if len(area_cols) > 0:
    area_col = area_cols[0]

    df_fe["log_area"] = np.log1p(
        pd.to_numeric(
            df_fe[area_col],
            errors="coerce"
        )
    )
if len(bedroom_cols) > 0:
    bedroom_col = bedroom_cols[0]

    df_fe["bedroom_squared"] = (
        pd.to_numeric(
            df_fe[bedroom_col],
            errors="coerce"
        ) ** 2
    )

if len(bathroom_cols) > 0:
    bathroom_col = bathroom_cols[0]

    df_fe["bathroom_squared"] = (
        pd.to_numeric(
            df_fe[bathroom_col],
            errors="coerce"
        ) ** 2
    )

print("Feature engineering completed")

display(df_fe.head())

In [ ]:
# train test Split
x=df.drop(columns=['Price_INR_Lakhs'],axis=1)
y=df['Price_INR_Lakhs']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
numeric_features = x_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = x_train.select_dtypes(
    exclude=np.number
).columns.tolist()

binary_cols = ["Parking", "Lift_Available", "Gated_Community"]


# Convert directly to boolean
df[binary_cols] = df[binary_cols].astype(bool)

numeric_features = [col for col in numeric_features if col not in binary_cols]


print("Numerical:", numeric_features)
print("\nCategorical:", categorical_features)
print("\nBinary:", binary_cols)

In [ ]:

for i in df.columns:
  print(i,df[i].nunique())

In [ ]:
# Create Pipeline
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [ ]:
categorical_transformer=Pipeline(
    steps=[
        ("encoder",OneHotEncoder(handle_unknown="ignore"),)
    ]
)

In [ ]:
x_train.shape

In [ ]:
# column  Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("bin", "passthrough", binary_cols),
        ("cat", categorical_transformer, categorical_features)
    ]
)
preprocessor

In [ ]:
# define regression model
models={
    'Linear Regression ': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=10),
    'Gradient Boosting Regressor': GradientBoostingRegressor(),
    'AdaBoost Regressor ': AdaBoostRegressor()
}
print('Model Created')
for name in models:
  print(name)

In [ ]:
# Build ML PipeLines
pipelines={}
for name,model in models.items():
  pipelines[name]=Pipeline(
      steps=[
          ('preprocessor',preprocessor),
          ('model',model)
      ]
  )
print('PipeLine is created')

In [ ]:
pipelines

In [ ]:
# train all Models
results = []

trained_models = {}

for name, pipeline in pipelines.items():

    print(f"Training: {name}")

    pipeline.fit(
        x_train,
        y_train
    )

    y_pred = pipeline.predict(
        x_test
    )

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    mse = mean_squared_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        y_pred
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = pipeline

print("All models trained!")

In [ ]:
# model Comparision
results_df=pd.DataFrame(results)
results_df

In [ ]:
# Find the exact row index where 'R2' is at its maximum
best_idx = results_df["R2"].idxmax()

# Extract the model name using .loc
best_model_name = results_df.loc[best_idx, "Model"]

# Get your trained model dictionary item
best_model = trained_models[best_model_name]

print("Best Model:")
print(best_model_name)
print("\nBest R2:", results_df.loc[best_idx, "R2"])

In [ ]:
# r2 comparision
plt.figure(figsize=(10,5))
sns.barplot(
    x='Model',
    y='R2',
    data=results_df
)
plt.xticks(rotation=45)
plt.show()

In [ ]:
best_model = trained_models[
    best_model_name
]

# model Prediction
best_predictions = best_model.predict(
    x_test
)

print("First 10 Predictions:")

display(
    pd.DataFrame({
        "Actual": y_test.iloc[:10].values,
        "Predicted": best_predictions[:10]
    })
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    best_predictions,
    alpha=0.5
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")

plt.title(
    f"Actual vs Predicted - {best_model_name}"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    best_predictions,
    alpha=0.5
)

minimum = min(
    y_test.min(),
    best_predictions.min()
)

maximum = max(
    y_test.max(),
    best_predictions.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")

plt.title("Actual vs Predicted")

plt.show()

In [ ]:
residuals = y_test - best_predictions

plt.figure(figsize=(10, 5))

sns.histplot(
    residuals,
    kde=True
)

plt.title("Residual Distribution")

plt.xlabel("Residual")

plt.show()

In [ ]:
mae = mean_absolute_error(
    y_test,
    best_predictions
)

mse = mean_squared_error(
    y_test,
    best_predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    best_predictions
)

print("FINAL MODEL PERFORMANCE")
print("=" * 40)

print("Model:", best_model_name)
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² Score:", r2)

In [ ]:
!pip install -q gradio

In [ ]:
best_model

In [ ]:
joblib.dump(
    best_model,
    "house_price_regression_pipeline.pkl"
)

print(
    "Model saved successfully!"
)

print(
    "house_price_regression_pipeline.pkl"
)

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib

model = joblib.load(
    "house_price_regression_pipeline.pkl"
)

print("Final model loaded successfully")

In [ ]:
input_components = []

for col in x_train.columns:




    if col in numeric_features:
        if col in ["Parking","Lift_Available","Gated_Community"]:
            component = gr.Checkbox(
                label=col,
                value=False
            )
        else:


            component = gr.Number(
            label=col,
            value=0
        )

    else:


        # Get example categorical value
        values = (
            x_train[col]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        component = gr.Dropdown(
            choices=values[:100],
            label=col
        )



    input_components.append(component)

print(
    "Number of Gradio inputs:",
    len(input_components)
)

In [ ]:
def predict_price(*inputs):

    input_data = {}

    for col, value in zip(
        x_train.columns,
        inputs
    ):
        input_data[col] = value

    input_df = pd.DataFrame(
        [input_data]
    )

    prediction = model.predict(
        input_df
    )[0]

    return float(prediction)

In [ ]:
demo = gr.Interface(

    fn=predict_price,

    inputs=input_components,

    outputs=gr.Number(
        label="Predicted House Price"
    ),

    title="🏠 Indian Urban House Price Prediction",

    description="""
    Machine Learning Regression Application

    Enter the property details below to predict
    the estimated house price.
    """,

    theme=gr.themes.Soft()
)

demo.launch(
    share=True
)